## Task 3: Exploratory Data Analysis (EDA)
Goal:
    Analyze the dataset to discover patterns and trends.
Key Requirements:
    • Calculate basic statistics
    • Identify trends and outliers
    • Summarize findings
Key Skills:
    Statistical thinking, data analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [2]:
plt.rcParams.update({
    "figure.facecolor": "#f5f5f0",
    "axes.facecolor":   "#f5f5f0",
    "axes.edgecolor":   "#4a6741",
    "axes.labelcolor":  "#3a5232",
    "xtick.color":      "#3a5232",
    "ytick.color":      "#3a5232",
    "text.color":       "#3a5232",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})
GREEN  = "#4a6741"
TEAL   = "#5b9b8a"
ORANGE = "#e07b39"
PALETTE = [GREEN, ORANGE, TEAL, "#8fbc8f", "#d4a76a"]
sns.set_palette(PALETTE)

In [3]:
df_raw = pd.read_csv("Titanic-Dataset.csv")
df = df_raw.copy()

In [4]:
# 1. Missing values BEFORE
print(f"\n Missing values BEFORE cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])


 Missing values BEFORE cleaning:
Age         177
Cabin       687
Embarked      2
dtype: int64


In [5]:
# 2. Drop Cabin (>77% missing — not salvageable)
df.drop(columns=["Cabin"], inplace=True)
print("\nDropped 'Cabin' column (77% missing, not recoverable)")


Dropped 'Cabin' column (77% missing, not recoverable)


In [6]:
# 3. Fill Age with median (robust to outliers)
age_median = df["Age"].median()
df["Age"].fillna(age_median, inplace=True)
print(f"Filled {df_raw['Age'].isnull().sum()} missing 'Age' values with median ({age_median})")

Filled 177 missing 'Age' values with median (28.0)


In [7]:
# 4. Fill Embarked with mode (only 2 missing)
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"].fillna(embarked_mode, inplace=True)
print(f"Filled 2 missing 'Embarked' values with mode ('{embarked_mode}')")

Filled 2 missing 'Embarked' values with mode ('S')


In [8]:
# 5. Remove duplicates
dupes_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {dupes_before}")

Duplicates removed: 0


In [9]:
# 6. Feature engineering — extract Title from Name
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df["Title"] = df["Title"].replace(
    ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"],
    "Rare"
)
df["Title"] = df["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
print("Extracted 'Title' feature from Name column")

Extracted 'Title' feature from Name column


In [10]:
# 7. Create FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
print("Created 'FamilySize' = SibSp + Parch + 1")

Created 'FamilySize' = SibSp + Parch + 1


In [11]:
# 8. Create IsAlone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
print("Created 'IsAlone' flag")

Created 'IsAlone' flag


In [12]:
# 9. Encode categoricals
df["Sex_enc"]      = LabelEncoder().fit_transform(df["Sex"])
df["Embarked_enc"] = LabelEncoder().fit_transform(df["Embarked"])
df["Title_enc"]    = LabelEncoder().fit_transform(df["Title"])
print("Label-encoded: Sex, Embarked, Title")

Label-encoded: Sex, Embarked, Title


In [13]:
# 10. Drop columns not needed for modelling
df.drop(columns=["PassengerId","Name","Ticket"], inplace=True)

In [14]:
print(f"\nMissing values AFTER cleaning: {df.isnull().sum().sum()} total")
print(f"Final dataset shape: {df.shape}")
print(f"\nSample cleaned rows:")
print(df.head(3).to_string())


Missing values AFTER cleaning: 0 total
Final dataset shape: (891, 14)

Sample cleaned rows:
   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked Title  FamilySize  IsAlone  Sex_enc  Embarked_enc  Title_enc
0         0       3    male  22.0      1      0   7.2500        S    Mr           2        0        1             2          2
1         1       1  female  38.0      1      0  71.2833        C   Mrs           2        0        0             0          3
2         1       3  female  26.0      0      0   7.9250        S  Miss           1        1        0             2          1


In [15]:
# ══════════════════════════════════════════════════════════
#  TASK 3 — EXPLORATORY DATA ANALYSIS (EDA)
# ══════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("  TASK 3 — EXPLORATORY DATA ANALYSIS (EDA)")
print("═"*60)

survival_rate = df["Survived"].mean() * 100
print(f"\nOverall Survival Rate: {survival_rate:.1f}%")


════════════════════════════════════════════════════════════
  TASK 3 — EXPLORATORY DATA ANALYSIS (EDA)
════════════════════════════════════════════════════════════

Overall Survival Rate: 38.4%


In [16]:
print("\n── Survival by Gender ───────────────────────────────────")
print(df.groupby("Sex")["Survived"].agg(["mean","count","sum"])
       .rename(columns={"mean":"Rate","count":"Total","sum":"Survived"})
       .assign(Rate=lambda x: (x["Rate"]*100).round(1))
       .to_string())



── Survival by Gender ───────────────────────────────────
        Rate  Total  Survived
Sex                          
female  74.2    314       233
male    18.9    577       109


In [17]:
print("\n── Survival by Passenger Class ──────────────────────────")
print(df.groupby("Pclass")["Survived"].agg(["mean","count","sum"])
       .rename(columns={"mean":"Rate","count":"Total","sum":"Survived"})
       .assign(Rate=lambda x: (x["Rate"]*100).round(1))
       .to_string())


── Survival by Passenger Class ──────────────────────────
        Rate  Total  Survived
Pclass                       
1       63.0    216       136
2       47.3    184        87
3       24.2    491       119


In [18]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'Title', 'FamilySize', 'IsAlone', 'Sex_enc', 'Embarked_enc',
       'Title_enc'],
      dtype='object')

In [19]:
print("\n── Survival by Title ────────────────────────────────────")
print(df.groupby("Title")["Survived"].agg(["mean","count"])
       .rename(columns={"mean":"Rate","count":"Total"})
       .assign(Rate=lambda x: (x["Rate"]*100).round(1))
       .sort_values("Rate", ascending=False)
       .to_string())


── Survival by Title ────────────────────────────────────
               Rate  Total
Title                     
the Countess  100.0      1
Mrs            79.4    126
Miss           70.3    185
Master         57.5     40
Rare           31.8     22
Mr             15.7    517


In [20]:
print("\n── Age Statistics by Survival ───────────────────────────")
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

# Survival rate within each age group
print(
    df.groupby("AgeGroup")["Survived"]
      .mean()
      .round(3)
      .to_string()
)


── Age Statistics by Survival ───────────────────────────
AgeGroup
Child          0.580
Teen           0.429
Young Adult    0.353
Adult          0.400
Senior         0.227


In [21]:
print("\n── Fare Statistics by Class ─────────────────────────────")
print(df.groupby("Pclass")["Fare"].describe().round(2).to_string())


── Fare Statistics by Class ─────────────────────────────
        count   mean    std  min    25%    50%   75%     max
Pclass                                                      
1       216.0  84.15  78.38  0.0  30.92  60.29  93.5  512.33
2       184.0  20.66  13.42  0.0  13.00  14.25  26.0   73.50
3       491.0  13.68  11.78  0.0   7.75   8.05  15.5   69.55


In [22]:
print("\n── Outlier Check (IQR method) ───────────────────────────")
for col in ["Age","Fare","FamilySize"]:
    Q1, Q3 = df[col].quantile([0.25,0.75])
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)).sum()
    print(f"  {col:<14}: {outliers} outliers detected")


── Outlier Check (IQR method) ───────────────────────────
  Age           : 66 outliers detected
  Fare          : 116 outliers detected
  FamilySize    : 91 outliers detected


In [23]:
print("\n── Key EDA Findings ─────────────────────────────────────")
findings = [
    "74.2% of women survived vs only 18.9% of men ('Women and children first')",
    "1st class had 63% survival rate vs 24% for 3rd class (wealth mattered)",
    "Children (age < 12) had higher survival rates",
    "Passengers with title 'Mrs'/'Miss' had highest survival rates",
    "Solo travelers (IsAlone=1) had lower survival than small families",
    "Fare has extreme outliers — max fare is 512 vs median ~14",
]
for i, f in enumerate(findings, 1):
    print(f"  {i}. {f}")


── Key EDA Findings ─────────────────────────────────────
  1. 74.2% of women survived vs only 18.9% of men ('Women and children first')
  2. 1st class had 63% survival rate vs 24% for 3rd class (wealth mattered)
  3. Children (age < 12) had higher survival rates
  4. Passengers with title 'Mrs'/'Miss' had highest survival rates
  5. Solo travelers (IsAlone=1) had lower survival than small families
  6. Fare has extreme outliers — max fare is 512 vs median ~14
